# Mapping Physicochemical Features onto 2D Interface Planes for Protein–Protein Binding Affinity Prediction

**Corso:** Advanced Machine Learning for Physics (A.A. 2025/2026)
**Istituzione:** Sapienza Università di Roma
**Candidata:** Chiara Ritorto

Questo notebook raccoglie ed esegue, in ordine, le fasi del progetto descritte nel report (`report/report.pdf`).
Il calcolo pesante (estrazione delle interfacce, ChimeraX, Zernike, training della CNN) non viene rilanciato qui:
il notebook carica i risultati già prodotti e ne mostra tabelle e grafici, in modo che sia eseguibile ovunque
(compreso Google Colab) senza dipendenze da ChimeraX o da un cluster HPC.


## 0. Setup dell'ambiente

In [ ]:
import os
import sys
from pathlib import Path

# Se il notebook gira su Google Colab, monta Drive e clona/aggiorna il repository.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_URL = "https://github.com/<tuo-utente>/ppb-affinity.git"  # <-- sostituisci con il link del tuo repo
    REPO_DIR = Path("/content/ppb-affinity")
    if not REPO_DIR.exists():
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
else:
    # Esecuzione locale: assume che il notebook sia nella root del repository
    REPO_DIR = Path.cwd()

sys.path.append(str(REPO_DIR))
print(f"Root del repository: {REPO_DIR}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

pd.set_option("display.max_columns", 20)
print("Librerie caricate correttamente.")


## 1. Dataset

Il dataset di partenza è l'*Affinity Benchmark v5.5*, con PDB ID, catene di ligando/recettore e costante di
dissociazione K$_D$ per ciascun complesso.

In [ ]:
df_affinity = pd.read_csv(REPO_DIR / "data" / "affinity_dataset.csv", sep=";")
df_affinity.columns = [c.strip() for c in df_affinity.columns]
print(f"Numero di complessi nel dataset: {len(df_affinity)}")
df_affinity.head()


## 2. Task 1 — Interface Identification and Surface Patch Extraction

Pipeline in `src/` (moduli `pdb.py`, `interface.py`, `surface.py`, `chimerax.py`, `dataset.py`, `pipeline.py`),
eseguita tramite `scripts/task1/run_interface_extraction.py`. Identifica i residui di interfaccia
(cutoff 5.0 Å) ed estrae le patch di superficie molecolare (backend ChimeraX, `full_chain_then_filter`).

Il risultato per ciascun complesso è salvato in `outputs/task1/<indice>_<PDB_ID>/`. Qui carichiamo il
`manifest.csv` riassuntivo e un complesso di esempio (1KTZ).

In [ ]:
manifest = pd.read_csv(REPO_DIR / "outputs" / "task1" / "manifest.csv")
print(f"Complessi nel manifest: {len(manifest)}")
print(manifest['status'].value_counts())
manifest.head()


### 2.1 Selezione dei dimeri

Dal dataset completo vengono selezionati i soli complessi dimerici (una catena per ligando e per recettore),
secondo lo stesso criterio usato in `scripts/task1/filter_dimers.py`.

In [ ]:
is_dimer = (df_affinity['Ligand Chains'].astype(str).str.len() == 1) & \
           (df_affinity['Receptor Chains'].astype(str).str.len() == 1)
dimer_ids = set(df_affinity.loc[is_dimer, 'PDB'].astype(str).str.strip())

manifest_dimers = manifest[manifest['pdb_id'].isin(dimer_ids)]
print(f"Complessi dimerici trovati: {len(manifest_dimers)}")


### 2.2 Esempio: complesso 1KTZ

In [ ]:
example_dir = REPO_DIR / "outputs" / "task1" / "00000_1KTZ"

with open(example_dir / "metadata.json") as f:
    meta_1ktz = json.load(f)

print(json.dumps(meta_1ktz["counts"], indent=2))


In [ ]:
ligand_patch = pd.read_csv(example_dir / "ligand_surface_patch.csv")
receptor_patch = pd.read_csv(example_dir / "receptor_surface_patch.csv")

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(ligand_patch["x"], ligand_patch["y"], ligand_patch["z"], s=2, alpha=0.5, label="ligando (A)")
ax.scatter(receptor_patch["x"], receptor_patch["y"], receptor_patch["z"], s=2, alpha=0.5, label="recettore (B)")
ax.set_title("Patch di interfaccia — complesso 1KTZ")
ax.legend()
plt.tight_layout()
plt.show()


### 2.3 Statistiche aggregate sui dimeri

In [ ]:
cols = ['ligand_interface_residues', 'receptor_interface_residues', 'residue_contacts',
        'ligand_patch_atoms', 'receptor_patch_atoms',
        'ligand_surface_points', 'receptor_surface_points', 'all_atoms']

manifest_dimers[cols].describe().T[['mean', 'std', 'min', '50%', 'max']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))

axes[0].hist(manifest_dimers['ligand_interface_residues'], bins=15, alpha=0.7, label='ligando', color='#4C72B0')
axes[0].hist(manifest_dimers['receptor_interface_residues'], bins=15, alpha=0.7, label='recettore', color='#DD8452')
axes[0].set_xlabel('Numero di residui di interfaccia')
axes[0].set_ylabel('Numero di complessi')
axes[0].set_title("Dimensione dell'interfaccia (dimeri)")
axes[0].legend(fontsize=8)

axes[1].hist(manifest_dimers['residue_contacts'], bins=15, color='#55A868')
axes[1].set_xlabel('Numero di contatti residuo-residuo')
axes[1].set_ylabel('Numero di complessi')
axes[1].set_title('Contatti di interfaccia (dimeri)')

plt.tight_layout()
plt.show()


## 3. Task 2 — Mappatura della complementarità tramite Zernike

*(sezione da popolare non appena saranno caricati gli output di produzione della Task 2)*

## 4. Task 3 — Costruzione dei piani di complementarità 2D

*(sezione da popolare non appena saranno caricati gli output della Task 3: mappe .npy, sommario PCA, analisi Lennard-Jones)*

## 5. Task 4 — Predizione dell'affinità tramite CNN

*(sezione da popolare non appena saranno caricati i pesi del modello e le predizioni out-of-fold)*

## 6. Discussione e conclusioni

*(da completare in linea con la Sezione "Discussione e limiti" del report)*